In [3]:
import pandas as pd
import json
from pathlib import Path

folder = Path("OPP-115/consolidation/threshold-0.75-overlap-similarity")
csv_files = list(folder.glob("*.csv"))
print(len(csv_files))
print(csv_files[:3])

115
[WindowsPath('OPP-115/consolidation/threshold-0.75-overlap-similarity/1017_sci-news.com.csv'), WindowsPath('OPP-115/consolidation/threshold-0.75-overlap-similarity/1028_redorbit.com.csv'), WindowsPath('OPP-115/consolidation/threshold-0.75-overlap-similarity/1034_aol.com.csv')]


In [4]:
cols = ["annotation_id", "batch_id", "annotator_id", "policy_id",
        "segment_id", "category", "attributes", "date", "url"]

all_dfs=[]  #empty list to catch each files dataframe

#iterate through all the csv files and tag each with the origin file(webpage name)
for file in csv_files:
    df = pd.read_csv(file, header=None, names=cols)
    df["source_file"] = file.name
    all_dfs.append(df)

#stack all dataframes into a single new vertically alligned DF
combined_df = pd.concat(all_dfs, ignore_index=True)
#shape check
print(combined_df.shape)
print(combined_df["source_file"].nunique())
print(combined_df["category"].value_counts())

(19720, 10)
115
category
First Party Collection/Use              8149
Third Party Sharing/Collection          4817
Other                                   2306
User Choice/Control                     1677
Data Security                            699
User Access, Edit and Deletion           690
International and Specific Audiences     529
Policy Change                            463
Data Retention                           337
Do Not Track                              53
Name: count, dtype: int64


In [5]:
#extracting each sites attribute names to check for irregularities
def get_attribute_names(attrs_json):
    return set(json.loads(attrs_json).keys())

#group by category and collect all unique attribute names for each
category_attrs = {}
for category, group in combined_df.groupby("category"):
    all_names = set()
    for attrs_json in group["attributes"]:
        all_names.update(get_attribute_names(attrs_json))
    category_attrs[category] = all_names

for cat, names in category_attrs.items():
    print(cat, "->", names)

Data Retention -> {'Personal Information Type', 'Retention Period', 'Retention Purpose'}
Data Security -> {'Security Measure'}
Do Not Track -> {'Do Not Track policy'}
First Party Collection/Use -> {'User Type', 'Collection Mode', 'Does/Does Not', 'Personal Information Type', 'Purpose', 'Identifiability', 'Action First-Party', 'Choice Scope', 'Choice Type'}
International and Specific Audiences -> {'Audience Type'}
Other -> {'Other Type'}
Policy Change -> {'User Choice', 'Change Type', 'Notification Type'}
Third Party Sharing/Collection -> {'User Type', 'Action Third Party', 'Third Party Entity', 'Personal Information Type', 'Purpose', 'Identifiability', 'Does/Does Not', 'Choice Scope', 'Choice Type'}
User Access, Edit and Deletion -> {'Access Scope', 'Access Type', 'User Type'}
User Choice/Control -> {'Personal Information Type', 'Purpose', 'User Type', 'Choice Scope', 'Choice Type'}


In [45]:
records= []

for i,row in combined_df.iterrows():
    attrs = json.loads(row["attributes"])
    category = row["category"]

    #default every row to None, only overwrites what appplies, will handle cleaning later
    purpose = None
    does_or_does_not = None
    entity = None

    if category == "First Party Collection/Use": #map first party category
        purpose = attrs.get("Purpose",{}).get("value") #using .get("attribute_name", {}) allows program to continue running if the .get("value") doesnt return anything, rather than giving keyerror
        does_or_does_not = attrs.get("Does/Does Not",{}).get("value")
    elif category == "Third Party Sharing/Collection": #map third party category
        purpose = attrs.get("Purpose",{}).get("value")
        does_or_does_not = attrs.get("Does/Does Not",{}).get("value")
        entity = attrs.get("Third Party Entity",{}).get("value")

    record_dict = {}
    record_dict["policy_id"] = row["policy_id"]
    record_dict["segment_id"] = row["segment_id"]
    record_dict["annotation_id"] = row["annotation_id"]
    record_dict["category"] = category
    record_dict["purpose"] = purpose
    record_dict["does_or_does_not"] = does_or_does_not
    record_dict["entity"] = entity
    records.append(record_dict)

df_check = pd.DataFrame(records)
print(df_check.shape)
print(df_check[df_check["category"] == "First Party Collection/Use"].head())
print(df_check[df_check["category"] == "Third Party Sharing/Collection"].head())
print(df_check["purpose"].value_counts(dropna=False))

print(df_check.head())

(19720, 7)
   policy_id  segment_id annotation_id                    category  \
3       3905           1         20324  First Party Collection/Use   
4       3905           1         20325  First Party Collection/Use   
5       3905           1         20590  First Party Collection/Use   
6       3905           1         20235  First Party Collection/Use   
7       3905           1         20236  First Party Collection/Use   

                 purpose does_or_does_not entity  
3  Basic service/feature             Does    NaN  
4  Basic service/feature             Does    NaN  
5            Unspecified             Does    NaN  
6  Basic service/feature             Does    NaN  
7            Unspecified             Does    NaN  
    policy_id  segment_id annotation_id                        category  \
30       3905           7         20597  Third Party Sharing/Collection   
31       3905           7         20248  Third Party Sharing/Collection   
35       3905           9         C27

In [43]:
fp_tp = df_check["category"].isin(["First Party Collection/Use", "Third Party Sharing/Collection"])
print(fp_tp.sum(), df_check[fp_tp]["purpose"].isna().sum())

cols_mapped = ["policy_id", "segment_id", "category", "purpose", "does_or_does_not", "entity"]
print(df_check.duplicated(subset=cols_mapped).sum())

12966 0
9823
